# 03_mining_or_clustering
Clustering, anomaly, rules


In [1]:
import sys, os, yaml
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))

from src.mining.association import compare_rules_by_season, save_seasonal_rules
from src.visualization.plots import plot_rules_scatter


In [2]:
# Load Data đã được xử lý (đã có Temp_Class, Humidity_Class...)
with open('../configs/params.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = pd.read_csv(f"../{config['processed_data_path']}")

In [3]:
# 1. Tìm luật
all_seasonal_rules = compare_rules_by_season(df, config)

In [4]:
# 2. Lưu luật và lấy thống kê
stats = save_seasonal_rules(all_seasonal_rules, config)

 Đã lưu luật tại: outputs/tables/association_rules_seasonal.csv


In [5]:
# 3. Trình bày Thống kê số lượng luật 
print("\n--- THỐNG KÊ SỐ LƯỢNG LUẬT THEO MÙA ---")
for season, count in stats.items():
    print(f"Mùa {season}: {count} luật tìm thấy")


--- THỐNG KÊ SỐ LƯỢNG LUẬT THEO MÙA ---
Mùa Spring: 50 luật tìm thấy
Mùa Summer: 114 luật tìm thấy
Mùa Autumn: 108 luật tìm thấy
Mùa Winter: 134 luật tìm thấy


In [6]:
# 4. Trực quan hóa (Ví dụ cho Mùa Đông và Mùa Hè để so sánh)
for season in ['Spring', 'Summer', 'Autumn', 'Winter']:
    print(f"\nĐồ thị phân tích luật mùa {season}:")
    plot_rules_scatter(all_seasonal_rules[season], season)


Đồ thị phân tích luật mùa Spring:



Đồ thị phân tích luật mùa Summer:



Đồ thị phân tích luật mùa Autumn:



Đồ thị phân tích luật mùa Winter:


Chúng ta có thể rút ra những quy luật khí hậu đặc trưng như sau.

Các chỉ số cần lưu ý:

Support (Hỗ trợ): Tần suất xuất hiện đồng thời của các điều kiện.

Confidence (Độ tin cậy): Xác suất xảy ra hệ quả khi đã có tiền đề.

Lift (Độ nâng): Nếu Lift > 1, các yếu tố có tác động tích cực đến nhau (quan hệ mạnh).

1. Mùa Xuân (Spring): Sự ấm áp và Khô ráo
Quy luật chủ đạo ở đây là mối quan hệ giữa Nhiệt độ ấm (Warm) và Độ ẩm thấp (Dry).

Diễn giải: Vào mùa xuân, khi tiết trời trở nên ấm áp (Temp_Class_Warm), độ ẩm thường có xu hướng giảm xuống mức khô (Humidity_Class_Dry).

Phân tích chi tiết: Các luật (3, 2, 17, 16, 19) đều cho thấy khi có mưa (Precip Type_rain), sự kết hợp giữa nhiệt độ ấm và độ ẩm khô vẫn rất chặt chẽ (Lift = 2.12). Điều này có thể hiểu là các đợt mưa xuân thường ngắn, sau đó thời tiết nhanh chóng trở lại trạng thái ấm khô.

Xác suất: Nếu thấy trời khô, có khoảng 63.4% khả năng là trời đang ấm.

2. Mùa Hạ (Summer): Nóng, Khô và Tầm nhìn trung bình
Mùa hè ghi nhận sự xuất hiện của các yếu tố Nhiệt độ cao (Hot) và Tầm nhìn trung bình (Medium).

Diễn giải: Đặc trưng rõ rệt nhất là khi trời Nóng và Tầm nhìn ở mức trung bình, thì chắc chắn đi kèm với Độ ẩm khô (Humidity_Class_Dry).

Phân tích chi tiết: Chỉ số Lift ở mùa hè rất cao (3.36), cao nhất trong các mùa. Điều này cho thấy sự liên kết cực kỳ bền vững giữa: Nóng + Tầm nhìn trung bình + Khô.

Ghi chú: Sự xuất hiện của "Visibility_Class_Medium" có thể do hiện tượng mù nhiệt hoặc bụi mịn khi thời tiết quá khô nóng.

3. Mùa Thu (Autumn): Ấm áp, Mưa và Tầm nhìn tốt
Mùa thu cho thấy sự chuyển giao với các yếu tố Nhiệt độ ấm (Warm), Độ ẩm bình thường (Normal) và Tầm nhìn cao (High).

Diễn giải: Quy luật phổ biến nhất là sự kết hợp giữa Mưa (Precip Type_rain), Độ ẩm bình thường và Tầm nhìn xa.

Phân tích chi tiết: Khác với mùa hè (khô), mùa thu có độ ẩm ở mức cân bằng hơn (Normal). Khi nhiệt độ ấm và tầm nhìn tốt, có đến 57.5% khả năng độ ẩm đang ở mức bình thường và có mưa.

Nhận xét: Đây là kiểu thời tiết điển hình của những ngày thu trong lành, dù có mưa nhưng không làm giảm tầm nhìn đáng kể.

4. Mùa Đông (Winter): Đóng băng, Tuyết và Tầm nhìn thấp
Đây là mùa có các quy luật khắc nghiệt và rõ ràng nhất.

Diễn giải: Quy luật "Vàng" của mùa đông là: Tuyết (Precip Type_snow) + Độ ẩm ướt (Wet) + Đóng băng (Freezing) + Tầm nhìn thấp (Low).

Phân tích chi tiết: * Luật 117 có độ tin cậy (Confidence) lên đến 99.5%. Nghĩa là: Nếu trời đang đóng băng và tầm nhìn thấp, gần như chắc chắn là đang có tuyết rơi và độ ẩm rất cao (ướt).

Tầm nhìn thấp là hệ quả trực tiếp của việc tuyết rơi dày và sương giá khi nhiệt độ xuống dưới 0 độ C.

Sự khác biệt: Ngoài ra còn một luật phụ (72) cho thấy nếu có mưa (rain) thay vì tuyết, thì nhiệt độ sẽ chỉ ở mức Lạnh (Cold) chứ không đến mức Đóng băng, và độ ẩm ở mức Bình thường.